# LifeMate: Agent + ToolEnv + evaluation trên Kaggle

Upload **chính notebook này**, Add Input dataset BTC, bật GPU rồi chạy từ trên xuống.
Notebook mang theo source Agent/evaluator cùng một snapshot; không cần clone GitHub
hoặc upload riêng `agent.py`. Không nhúng dataset, checkpoint hay credentials.

Mặc định: **10 mẫu validation lấy từ train**, transcript chuẩn current/history
-> Qwen thật -> SDK ToolEnv thật -> validate/replay -> điểm ước lượng và diagnostics.
`SAMPLE_COUNT=1` để smoke test; `None` để chạy toàn validation. Đây không phải public test.
Gold calls/final state chỉ dùng sau inference. Oracle chỉ là chế độ kiểm tra riêng Agent;
notebook không chạy ASR model.

## 1. Cấu hình

`DATA_ROOT=None` tự tìm dataset đã Add Input. `MODEL_SOURCE` có thể là thư mục
checkpoint có `config.json` hoặc model ID Hugging Face. Model ID cần cache hoặc
Internet khi `ALLOW_MODEL_DOWNLOADS=True`. Để tái lập model, điền `MODEL_REVISION`.

In [ ]:
from pathlib import Path

DATA_ROOT = None
SAMPLE_COUNT = 10       # 1: smoke; 10: thử ngắn; None: toàn validation
SAMPLE_IDS = None       # Tuỳ chọn: IDs thuộc validation; không giả định IDs liên tục
ASR_MODE = "oracle_text" # "oracle_text" hoặc "asr_text"
SPEECH_FILE = None      # asr_text: file asr_predictions.jsonl gồm current + history

MODEL_SOURCE = "Qwen/Qwen3-4B-Instruct-2507"
MODEL_REVISION = None   # Commit checkpoint HF; None phải được ghi nhận là chưa pin
DEVICE = "cuda:0"
ALLOW_MODEL_DOWNLOADS = True
INSTALL_DEPENDENCIES = True  # Chỉ cài khi thiếu/sai phiên bản; giữ PyTorch/CUDA Kaggle
AGENT_OPTIONS = {}      # Mặc định 48 model steps, 12 tool calls; không đổi thuật toán

WORK_ROOT = Path("/kaggle/working/lifemate-unified-evaluation")
SOURCE_WORK_ROOT = Path("/kaggle/working/lifemate-unified-source")
LAST_RESULT = None

## 2. Cài source đi cùng notebook

Cell này chỉ giải nén source/config, kiểm tra SHA256 và chọn đúng đường import.
Nếu kernel đã import LifeMate từ notebook/source khác, **Restart Session** trước.
Hash in ra là danh tính source thực chạy, không chỉ là Git commit.

In [ ]:
# The uploaded notebook carries this exact source/config snapshot.
import base64
import hashlib
import importlib
import io
import json
from pathlib import Path
import sys
import zipfile

SOURCE_BASE_COMMIT = 'ddcbf55abd7d171a672e885ad7bead2c242a3afd'
SOURCE_BUNDLE_SHA256 = '56c11763bacc092575ca0ebbfcdca5b8dcdaca46f6baaf6ec4b2dfc906249e49'
SOURCE_FILES_SHA256 = {'src/lifemate/__init__.py': '80cff9a952c45f34c6c25e5b7732b656388b8d38098708fdd7e7d06b911cb4d7', 'src/lifemate/agent.py': '7c04347169b5c0219efd9774f98a0a3f0e5e5eea864247a894ed9c28476a543e', 'src/lifemate/asr/__init__.py': 'f714e61f37a84abe6d1f41ad30cd663bdf6488f8c0c25819eff81851c241135a', 'src/lifemate/asr/predict.py': 'd5f549b8df626b04a2597799bbe57d31b4bafc8b521dbef141a2b737ce3fef20', 'src/lifemate/asr/predictions.py': 'cea2cb6b2706979350987dbd474faca872cd7ff4d7efdc4a0603b78e7a3e09eb', 'src/lifemate/evaluation/__init__.py': 'e7b77ca25dc42d73debd0718032ed40baf07d54f1970a079ed0e4b6f0fd59b07', 'src/lifemate/evaluation/__main__.py': 'b58d35a762750f9b7369f3fade73aa10faa3afca52ca2984ea4a2288dd822a46', 'src/lifemate/evaluation/compare.py': '98ece6a0d39316b754a4dcb9cbaf4d35b095c69842bfe65cf5031d6c62d5d6da', 'src/lifemate/evaluation/data.py': '16c25c5b9543d88ec0dd2261d71aa4d78ff8963c8fac91682d3baa42dcf76c86', 'src/lifemate/evaluation/evaluate.py': '8d33b0b3b923fbfdf1c5e0c3bafa6ccb06d50f6c8cd219a25445d9ddf3989bd0', 'src/lifemate/evaluation/experiment.py': 'cae072f5d78505682ba7c942d826c3463c0ed5caeab9e23dafc682fcd4cd9c6c', 'src/lifemate/evaluation/io.py': '78aee4c13e0ccfd633c8f384e3200f688db606cfa085c5293eb65f7fecc4bf1f', 'src/lifemate/evaluation/matching.py': '804add29e2783b07226c01c3114b7d1b23c68555c6795489f8bbf07de276abd5', 'src/lifemate/evaluation/runner.py': 'd4635f98f034f10f093d3acdd92d8350e52cb545c6dcd3ebabe2f5d1398e7f84', 'src/lifemate/evaluation/runtime.py': 'ae530969741a71a0be17e40cf09c3fe8bc3e12f4d6fc3f4c7a448a093b465f03', 'src/lifemate/evaluation/selection.py': 'dc64e09bf3c6af69e86f56246c845437fd373209d08af41cd93f25272c3a16f9', 'src/lifemate/evaluation/validate.py': 'c580813d533ed8014997bd7bb80c4a638efad9e85b81490a83cf67cd6b5d2fd4', 'src/lifemate/paths.py': '9418dd9a2026581a0320ce73dc3d5c54ef4a0f9c046a57addd84e464bd59e314', 'src/lifemate/pipeline/__init__.py': '7a11e8880da416deda8edfd9206ee06ea002720812ec9fd91b775c1093ee5fa5', 'src/lifemate/pipeline/actor.py': '91c3a179c281f409784aa7bbc70688d621efec513c3e3f6f00ca011f36f8abdf', 'src/lifemate/pipeline/adaptive.py': '799d16b397892716d4ec4655f577d415c7a929f4245ef6cd723d492e19ac536f', 'src/lifemate/pipeline/backend.py': '13f527c8e646be9f5905b80649fcfb728ad01b00d97eea8d0bc3cbe6347d4d93', 'src/lifemate/pipeline/call_repair.py': 'b724754e5d5fadb350e920a82deaf3aa44cc337772c011dcee1699fcd531094f', 'src/lifemate/pipeline/checkpoint.py': '4f3f828e7717c01a3841094ce9537d8a3ebb431bccdeb5dd3a41e6674dd7bca7', 'src/lifemate/pipeline/config.py': '50a2b9afad7f98e7b7e9a62e779027e3da28b4c75c1cc830a249d4fe7a2f1cc3', 'src/lifemate/pipeline/controller.py': 'd6b96828ce22dc9debdbc85e9fc32f734efacbd4cc480e089f52a4001ba2f3ed', 'src/lifemate/pipeline/execution.py': 'e7eee4374f73217d2c91f9405b816579c4952814ccd1725270b14f68a7f540d3', 'src/lifemate/pipeline/grounding.py': 'a7b3d45eef0f9e5af70fedb9be8d3b836719ed7c73acd33fd186c7650e27a0cb', 'src/lifemate/pipeline/inputs.py': '4d5373f0d94fb84544a2593d6f101cec092ec0934b4ebe0115b012767ad59f24', 'src/lifemate/pipeline/messages.py': 'ef997ed9cf7ddb3c84f8a79c630b21d45d836c61713a33b98ff2263b9ea19d96', 'src/lifemate/pipeline/planning.py': '6b6a1efe2cf6b6b2fea40028b2ea7fe054367c2e0492fe5540c5cce69de0a60b', 'src/lifemate/pipeline/predicates.py': '466e6a41db54cf6272a16b37a5b647152fce1e846bea798b42087170abafcc46', 'src/lifemate/pipeline/proposals.py': 'e5ce0cf1e63bdb22cac9fb5c9172a4e16de67864d9c63d7e1f5f92dcbf8563a7', 'src/lifemate/pipeline/protocol.py': 'abc3912b201b36b4574e44669c3c36e7e14f15a31caf0c5466ce276a87980c1d', 'src/lifemate/pipeline/quantities.py': '2a0d2dd7e9382852591739d8f8f9e8f83a86f97152965792065b9f23c7dab47f', 'src/lifemate/pipeline/recovery.py': '8ee38f3c99ae51241cc9f5ac7ba0e5e147e45381f98c8f794665089a78531a41', 'src/lifemate/pipeline/replies.py': '3f6c624bde3f40f24e898a2d41b32d6a9bf57a7af6c55285c7b100505476d2db', 'src/lifemate/pipeline/results.py': '91d976490557e59642e9f8df4c57a6a0c45f21615564e83f9b72a50466f580c8', 'src/lifemate/pipeline/runner.py': '4252a5052170f2ad54877c1b9aa3aa10e39853af3d54c7c716bda45a86ed5c81', 'src/lifemate/pipeline/state.py': '58575183208c0e4c91cbe4b3ba353a763c6d327c8b4aec570d42612c9c4884f6', 'src/lifemate/pipeline/task_reasoning.py': '7d5320352e4766e053dfb5dcd8aaca312b494b102e57e9beb553835ebdb6555b', 'src/lifemate/pipeline/tools.py': 'fc8f911c13f78a7b1e491c9a5ea4581248cb84c9c11cc7030f13cfb43de795f4', 'src/lifemate/train_eval.py': '24d234213790abbde9c6bacd63320d604358de6eba9f8a422b045ab79a212666', 'src/lifemate/workflow.py': 'c372ba2f0f0ed1b3eeaa81eab9bc8baf83c1ac44858665921720f707d04769fd', 'configs/lifemate/agent-evaluation.json': '2cab4a84044226e7864b120e4e3214868401bc947b9d8e51b58024256a1e2f3e'}
SOURCE_ARCHIVE_BASE64 = '...'

def install_embedded_source(work_root):
    archive_bytes = base64.b64decode(SOURCE_ARCHIVE_BASE64, validate=True)
    if hashlib.sha256(archive_bytes).hexdigest() != SOURCE_BUNDLE_SHA256:
        raise RuntimeError("Embedded source archive checksum mismatch")
    target = Path(work_root).resolve() / SOURCE_BUNDLE_SHA256
    for name, loaded in list(sys.modules.items()):
        if loaded is None or not (name == "lifemate" or name.startswith("lifemate.")):
            continue
        origin = getattr(loaded, "__file__", None)
        if origin is None or not Path(origin).resolve().is_relative_to(target / "src/lifemate"):
            raise RuntimeError("Kernel already imported another LifeMate source. Restart Session, then Run All.")
    with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
        if (len(archive.namelist()) != len(SOURCE_FILES_SHA256)
                or set(archive.namelist()) != set(SOURCE_FILES_SHA256)):
            raise RuntimeError("Unexpected files in embedded source archive")
        verified = []
        for name, expected in SOURCE_FILES_SHA256.items():
            path = (target / name).resolve()
            data = archive.read(name)
            if not path.is_relative_to(target) or hashlib.sha256(data).hexdigest() != expected:
                raise RuntimeError("Embedded source file checksum mismatch: " + name)
            if path.exists() and hashlib.sha256(path.read_bytes()).hexdigest() != expected:
                raise RuntimeError("Source file was changed; refusing to overwrite: " + str(path))
            verified.append((path, data))
        for path, data in verified:
            path.parent.mkdir(parents=True, exist_ok=True)
            if not path.exists():
                with path.open("xb") as stream:
                    stream.write(data)
    source_path = str(target / "src")
    if source_path not in sys.path:
        sys.path.insert(0, source_path)
    importlib.invalidate_caches()
    return target

PROJECT_ROOT = install_embedded_source(SOURCE_WORK_ROOT)
print("Source root:", PROJECT_ROOT)
print("Source bundle SHA256:", SOURCE_BUNDLE_SHA256)
print("Base Git commit (source may include local edits):", SOURCE_BASE_COMMIT)

## 3. Chuẩn bị selection và kiểm tra cấu hình trước model

Chia tập train theo grouping/seed cố định, chọn subset validation và dùng cùng selection
cho inference lẫn evaluator. Chuẩn bị lần đầu có thể mất thời gian vì kiểm tra audio.
Không tải/nạp Qwen tại bước này. Mỗi lần chạy cell tạo một thư mục thí nghiệm mới;
bundle chia dữ liệu đã kiểm tra được dùng lại dưới cùng WORK_ROOT.

In [ ]:
plan = None
LAST_RESULT = None
import json

from lifemate.agent import discover_data_root
from lifemate.evaluation.experiment import prepare_experiment

if ASR_MODE not in {"oracle_text", "asr_text"}:
    raise ValueError("ASR_MODE phải là oracle_text hoặc asr_text")
if ASR_MODE == "asr_text" and SPEECH_FILE is None:
    raise ValueError("Đặt SPEECH_FILE tới file ASR current/history của đội")
dataset_root = discover_data_root(explicit=DATA_ROOT) if DATA_ROOT else discover_data_root()
plan = prepare_experiment(
    dataset_root, WORK_ROOT, count=SAMPLE_COUNT, sample_ids=SAMPLE_IDS,
    speech_path=SPEECH_FILE, oracle=ASR_MODE == "oracle_text",
    agent_options=AGENT_OPTIONS, model_source=MODEL_SOURCE,
    model_revision=MODEL_REVISION, device=DEVICE,
)
effective_settings = json.loads(Path(plan["config"]).read_text(encoding="utf-8"))
print("Dataset root:", dataset_root)
print("ASR source:", ASR_MODE)
print("Selected validation samples:", len(plan["selected_ids"]))
print("Selected IDs:", plan["selected_ids"])
print("Agent options:", effective_settings["generation"])
print("Selection manifest:", plan["selection"])
print("Experiment:", plan["output_dir"])

## 4. Dependencies của kernel Kaggle

Chỉ cài Transformers/Accelerate/HF Hub khi cần. Không cài lại Torch/CUDA hoặc ASR SDK.
Nếu thư viện cũ đã được import trước khi thay phiên bản, restart session rồi Run All.

In [ ]:
import importlib.metadata
import subprocess
import sys

def installed_version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None

def compatible_major_minor(value, minimum):
    if value is None:
        return False
    try:
        parts = tuple(int(part) for part in value.split(".")[:2])
        return minimum <= parts < (2, 0)
    except ValueError:
        return False

versions = {name: installed_version(name) for name in
            ("torch", "transformers", "accelerate", "huggingface-hub")}
needs_install = (versions["transformers"] != "5.17.0"
                 or not compatible_major_minor(versions["accelerate"], (1, 12))
                 or not compatible_major_minor(versions["huggingface-hub"], (1, 5)))
if needs_install and INSTALL_DEPENDENCIES:
    if globals().get("qwen_backend") is not None or any(
            name in sys.modules for name in ("transformers", "accelerate", "huggingface_hub")):
        raise RuntimeError("Dependencies need updating after import. Restart Session, then Run All.")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet",
                           "transformers==5.17.0", "accelerate>=1.12,<2", "huggingface-hub>=1.5,<2"])
elif needs_install:
    raise RuntimeError("Dependencies chưa phù hợp; đặt INSTALL_DEPENDENCIES=True và chạy lại cell này")
print("Dependencies:", {name: installed_version(name) for name in versions})

## 5. Nạp Qwen một lần

Cell này cần GPU. Nếu cùng source/revision/device đã được nạp, dùng lại `qwen_backend`.
Khi thay model/device, restart session để tránh giữ hai bộ weights trong GPU.

In [ ]:
import torch
import transformers
from transformers import GenerationConfig
from lifemate.agent import AgentConfig, ModelLoadConfig, load_qwen_backend

if not torch.cuda.is_available():
    raise RuntimeError("Bật GPU trong Settings của Kaggle trước khi nạp Qwen")
if not hasattr(GenerationConfig(), "prefill_chunk_size"):
    raise RuntimeError("Transformers thiếu prefill_chunk_size; kiểm tra cell dependencies và restart")
model_settings = json.loads(Path(plan["config"]).read_text(encoding="utf-8"))
model_source = model_settings["model_source"]
model_revision = model_settings.get("model_revision")
model_device = model_settings.get("device", "cuda:0")
backend_key = (model_source, model_revision, model_device)
if globals().get("qwen_backend") is not None:
    if globals().get("loaded_qwen_key") != backend_key:
        raise RuntimeError("Đã có Qwen khác trong kernel; restart trước khi đổi model/revision/device")
    print("Reusing Qwen backend:", loaded_qwen_key)
else:
    qwen_model_config = ModelLoadConfig(
        model_source=model_source, device=model_device, revision=model_revision,
        local_files_only=not ALLOW_MODEL_DOWNLOADS,
    )
    qwen_backend = load_qwen_backend(
        qwen_model_config, AgentConfig(**model_settings["generation"]))
    loaded_qwen_key = backend_key
print("GPU:", torch.cuda.get_device_name(int(model_device.split(":")[1])))
print("Torch:", torch.__version__, "Transformers:", transformers.__version__)
print("Requested model revision:", model_revision)
print("Loaded model commit:", getattr(getattr(getattr(qwen_backend, "model", None), "config", None), "_commit_hash", None))

## 6. Chạy Agent, ToolEnv và evaluation

Đây là cell chạy model. Cùng selection được dùng cho cả Agent và scorer; không chấm
10 predictions trên mẫu số của toàn validation. Mỗi mẫu có một inference ToolEnv.
Evaluator replay trong env riêng, không reset hoặc thay state của Agent.
Chạy lại bằng một `plan` mới (cell 3); không ghi đè/chọn lại kết quả của run cũ.

In [ ]:
LAST_RESULT = None
if globals().get("qwen_backend") is None:
    raise RuntimeError("Chạy cell 5 để nạp Qwen trước")
if globals().get("plan") is None:
    raise RuntimeError("Chạy cell 3 để tạo selection/config trước")
import json
from pathlib import Path
planned = json.loads(Path(plan["config"]).read_text(encoding="utf-8"))
expected_backend = (planned["model_source"], planned.get("model_revision"),
                    planned.get("device", "cuda:0"))
if globals().get("loaded_qwen_key") != expected_backend:
    raise RuntimeError("Model đang nạp không khớp plan mới; chạy cell 5 hoặc restart trước khi đổi model")
from lifemate.evaluation.experiment import run_experiment

LAST_RESULT = run_experiment(plan, backend=qwen_backend)
print("Summary:", LAST_RESULT["summary_path"])

## 7. Báo cáo tổng hợp và tải kết quả

Điểm local dựa trên calls/hash và convention được ghi trong report. Status runtime
(`finished`, `waiting_for_user`, `step_limit`...) được báo riêng; `finished` không chứng minh đúng.
`submitted_hash == replay_hash` kiểm tra pipeline, còn `replay_hash == gold_hash` đo trạng thái đích.

In [ ]:
from IPython.display import FileLink, display

if LAST_RESULT is None:
    raise RuntimeError("Chưa có run hoàn tất; xem lỗi ở cell 6")
summary = json.loads(Path(LAST_RESULT["summary_path"]).read_text(encoding="utf-8"))
print(json.dumps(summary, ensure_ascii=False, indent=2))
artifacts = [Path(LAST_RESULT["summary_path"]), Path(LAST_RESULT["predictions"]),
             Path(LAST_RESULT["traces"])]
evaluation_dir = Path(LAST_RESULT["evaluation_dir"])
artifacts += [evaluation_dir / name for name in ("metrics.json", "samples.jsonl", "report.md")]
artifacts += [Path(plan[name]) for name in ("selection", "config")]
if LAST_RESULT.get("validation_dir"):
    artifacts.append(Path(LAST_RESULT["validation_dir"]) / "validation_summary.json")
elif LAST_RESULT.get("output_dir"):
    artifacts.append(Path(LAST_RESULT["output_dir"]) / "validation/validation_summary.json")
for path in artifacts:
    if path.is_file():
        display(FileLink(str(path)))

## 7b. In đầy đủ ba file evaluation

Chạy cell dưới sau mục 6 để in `metrics.json`, toàn bộ `samples.jsonl` và
`report.md` của đúng run. Cell chỉ đọc file, không chạy lại Qwen hay chấm lại.
Nếu mở kết quả đã lưu sau restart, đặt `EVALUATION_DIR` tới thư mục có ba file.

In [ ]:
from pathlib import Path
import json

EVALUATION_DIR = None  # Hoặc "/kaggle/working/.../evaluation" cho run đã lưu
latest_result = globals().get("LAST_RESULT")
if EVALUATION_DIR is not None:
    evaluation_path = Path(EVALUATION_DIR)
elif latest_result and latest_result.get("evaluation_dir"):
    evaluation_path = Path(latest_result["evaluation_dir"])
else:
    raise RuntimeError("Chạy mục 6 trước, hoặc đặt EVALUATION_DIR tới thư mục evaluation đã lưu")

filenames = ("metrics.json", "samples.jsonl", "report.md")
for filename in filenames:
    if not (evaluation_path / filename).is_file():
        raise FileNotFoundError(evaluation_path / filename)
print("Evaluation directory:", evaluation_path)
for filename in filenames:
    content = (evaluation_path / filename).read_text(encoding="utf-8")
    print("\n==========", filename, "==========")
    if filename.endswith(".json"):
        print(json.dumps(json.loads(content), ensure_ascii=False, indent=2))
    elif filename.endswith(".jsonl"):
        for line in content.splitlines():
            if line.strip():
                print(json.dumps(json.loads(line), ensure_ascii=False, indent=2))
    else:
        print(content)

## 8. Mẫu lỗi và phân biệt các chỉ số

Xem `score`, `H` (hash khớp gold), `M/K/G` (matched/submitted/gold calls), status và lỗi.
Một mẫu runtime dừng sớm vẫn được chấm từ submission thật nếu hợp lệ/replay được.
Mẫu thiếu prediction hoặc hash không tái lập được không được giả thành danh sách rỗng đúng.

In [ ]:
from collections import Counter

if LAST_RESULT is None:
    raise RuntimeError("Chạy cell 6 trước")
sample_reports = [json.loads(line) for line in
                  (Path(LAST_RESULT["evaluation_dir"]) / "samples.jsonl").read_text(
                      encoding="utf-8").splitlines() if line.strip()]
print("Statuses:", dict(Counter(row.get("runtime_status") for row in sample_reports)))
print("Evaluated samples:", len(sample_reports))
for row in [r for r in sample_reports if r.get("score", 0) < 1][:10]:
    print(row["id"], "score=", row["score"], "H=", row["H"],
          "M/K/G=", (row["M"], row["K"], row["G"]),
          "status=", row.get("runtime_status"), "errors=", row.get("observed_errors"))

## 8b. Mở rộng thử nghiệm

- Smoke: đặt `SAMPLE_COUNT=1`, chạy cell 3 rồi 5 -> 6 -> 7 -> 8. Cell 5 dùng lại model đã nạp.
- Thử ngắn: `SAMPLE_COUNT=10`; toàn validation: `SAMPLE_COUNT=None`.
- Đo sau ASR: `ASR_MODE="asr_text"`, đặt `SPEECH_FILE`, giữ cùng model/config/IDs.
- Khi sửa code local, chạy `python scripts/build_lifemate_test_notebook.py`, upload lại notebook
  mới và restart session. Sửa file local không tự cập nhật notebook đã upload.

Phép đo này không bao gồm thời gian nhận dạng ASR và chưa được đối chiếu scorer BTC chính thức.
Giữ nguyên convention local `token-F1 > 0.6`, F1 calls `2M/(K+G)` theo hình đã cung cấp;
tài liệu text trong repo có mâu thuẫn được ghi rõ trong báo cáo.

In [ ]:
# Chỉ đọc đường dẫn, không chạy model/evaluator lại.
if globals().get("plan"):
    print("Selected IDs:", plan["selected_ids"])
    print("Config:", plan["config"])
    print("Selection:", plan["selection"])
print("Source bundle:", SOURCE_BUNDLE_SHA256)

## 9. Trace chi tiết — tuỳ chọn

Điền `TRACE_SAMPLE_ID` rồi chạy cell này để đọc các event của đúng run.
Trace chứa request/tool arguments của dataset; giữ kết quả private khi chia sẻ notebook.
Cell này chỉ đọc log, không gọi model hoặc thực thi lại tool.

In [ ]:
# Có thể đặt hai biến này trong cell trước để đọc kết quả đã lưu sau restart.
TRACE_SAMPLE_ID = globals().get("TRACE_SAMPLE_ID")
TRACE_RESULTS_FILE = globals().get("TRACE_RESULTS_FILE")
if TRACE_SAMPLE_ID is None:
    print("Điền TRACE_SAMPLE_ID để xem chi tiết một mẫu")
else:
    from pathlib import Path
    import json
    latest = globals().get("LAST_RESULT")
    trace_path = TRACE_RESULTS_FILE or (latest.get("traces") if latest else None)
    if trace_path is None:
        raise RuntimeError("Đặt TRACE_RESULTS_FILE tới results.jsonl đã lưu, hoặc chạy cell 6")
    print("Trace file:", trace_path)
    rows = [json.loads(line) for line in Path(trace_path).read_text(
        encoding="utf-8").splitlines() if line.strip()]
    selected_trace = next((r for r in rows if r.get("id") == TRACE_SAMPLE_ID), None)
    if selected_trace is None:
        raise ValueError("ID không thuộc run hiện tại: " + str(TRACE_SAMPLE_ID))
    print("ID:", TRACE_SAMPLE_ID, "STATUS:", selected_trace.get("status"))
    for event in selected_trace.get("trace", []):
        print(json.dumps(event, ensure_ascii=False, indent=2))